In [1]:
import numpy as np
from matplotlib import pyplot as plt

In [2]:
users = []
movies = []
ratings = []
with open("../week1/movieLense-100k/ratings.csv") as f:
    for line in f:
        temp = (line.strip().split(","))
        users.append(temp[0])
        movies.append(temp[1])
        ratings.append(temp[2])

users.pop(0)
movies.pop(0)
ratings.pop(0)
print(len(users))
print(len(movies))
print(len(ratings))

print(f"{len(users)} x {len(movies)}")

100836
100836
100836
100836 x 100836


In [24]:
def softImputeALS(Matrix, observation, lam, rank=40, max_iter=500, tol=1e-5, seed=42):
    Matrix = Matrix.astype(float)

    rng = np.random.default_rng(seed)

    # random orthonormal user factors, as in the paper
    A = rng.normal(size=(Matrix.shape[0], rank))
    A, _ = np.linalg.qr(A)

    # movie factors begin at zero
    B = np.zeros((Matrix.shape[1], rank))

    oldEstimate = A @ B.T
    I = np.eye(rank)

    for i in range(max_iter):

        # fill missing entries using current prediction
        Xstar = np.where(observation, Matrix, A @ B.T)

        # update movie factors B
        B = Xstar.T @ A @ np.linalg.inv(A.T @ A + lam * I)

        # fill again using newest B
        Xstar = np.where(observation, Matrix, A @ B.T)

        # update user factors A
        A = Xstar @ B @ np.linalg.inv(B.T @ B + lam * I)

        newEstimate = A @ B.T

        change = np.linalg.norm(newEstimate - oldEstimate) ** 2 / max(np.linalg.norm(oldEstimate) ** 2, 1e-12)

        if i % 10 == 0:
            print(f"iteration {i}, change = {change:.6f}")

        if change < tol:
            print(f"converged after {i + 1} iterations")
            break

        oldEstimate = newEstimate

    return newEstimate

In [25]:
uniqueUsers = sorted(set(users), key=int)
uniqueMovies = sorted(set(movies), key=int)

userIndex = {}
movieIndex = {}

for i, user in enumerate(uniqueUsers):
    userIndex[user] = i

for i, movie in enumerate(uniqueMovies):
    movieIndex[movie] = i


Matrix = np.zeros(
    (len(uniqueUsers), len(uniqueMovies))
)

observation = np.zeros(
    Matrix.shape,
    dtype=bool
)


for user, movie, rating in zip(users, movies, ratings):

    u = userIndex[user]
    m = movieIndex[movie]

    Matrix[u][m] = float(rating)

    observation[u][m] = True


print("Matrix shape:", Matrix.shape)
print("Observed ratings:", np.sum(observation))

Matrix shape: (610, 9724)
Observed ratings: 100836


In [26]:
rows, cols = np.where(observation)
indices = np.arange(len(rows))

rng = np.random.default_rng(42)
rng.shuffle(indices)

trainEnd = int(len(indices) * 0.90)
validationEnd = int(len(indices) * 0.95)

trainIndices = indices[:trainEnd]
validationIndices = indices[trainEnd:validationEnd]
testIndices = indices[validationEnd:]

trainObservation = np.zeros(Matrix.shape, dtype=bool)
validationObservation = np.zeros(Matrix.shape, dtype=bool)
testObservation = np.zeros(Matrix.shape, dtype=bool)

trainObservation[rows[trainIndices], cols[trainIndices]] = True
validationObservation[rows[validationIndices], cols[validationIndices]] = True
testObservation[rows[testIndices], cols[testIndices]] = True

print("Train:", np.sum(trainObservation))
print("Validation:", np.sum(validationObservation))
print("Test:", np.sum(testObservation))

Train: 90752
Validation: 5042
Test: 5042


In [27]:
def getRMSE(Matrix, prediction, observation):
    rows, cols = np.where(observation)
    errors = [(Matrix[row][col] - prediction[row][col]) ** 2 for row, col in zip(rows, cols)]
    return np.sqrt(np.mean(errors))

In [28]:
lambdaValues = [1e3, 1e2, 1e1, 1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]

In [29]:
prediction = softImputeALS(Matrix, trainObservation, lam=10, rank=40)

validationRMSE = getRMSE(Matrix, prediction, validationObservation)
testRMSE = getRMSE(Matrix, prediction, testObservation)

print("Validation RMSE:", validationRMSE)
print("Test RMSE:", testRMSE)

iteration 0, change = 289740962029828032.000000
iteration 10, change = 0.000904
iteration 20, change = 0.000246
iteration 30, change = 0.000132
iteration 40, change = 0.000085
iteration 50, change = 0.000059
iteration 60, change = 0.000042
iteration 70, change = 0.000032
iteration 80, change = 0.000025
iteration 90, change = 0.000019
iteration 100, change = 0.000016
iteration 110, change = 0.000013
iteration 120, change = 0.000011
converged after 126 iterations
Validation RMSE: 1.3743185044270925
Test RMSE: 1.38940851648034
